<a href="https://colab.research.google.com/github/JeissonCu/TareasPucp/blob/main/final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Capstone Project - Deep Learning
## TEC-VIII Programa de Especialización en Big Data Analytics aplicada a los Negocios

---

### 📋 Información del Proyecto

| Campo | Información |
|-------|-------------|
| **Nombre del Estudiante** | Cueva Caro Jeisson Enrique (Grupo N.º 1) |
| **Título del Proyecto** | Detección de Anomalías y Fraude en el Sistema Nacional de Pensiones — Auditoría no supervisada y supervisada mediante Deep Learning (Autoencoder + MLP Clasificador) |
| **Fecha de Entrega** | Agosto 2026 |
| **Profesor** | C. Marino Del Rosario, Ph.D. |

---


## 📑 Índice

1. [Resumen Ejecutivo](#1-resumen-ejecutivo)
2. [Configuración del Entorno](#2-configuración-del-entorno)
3. [Definición del Problema de Negocio](#3-definición-del-problema-de-negocio)
4. [Carga y Exploración de Datos](#4-carga-y-exploración-de-datos)
5. [Preprocesamiento de Datos](#5-preprocesamiento-de-datos)
6. [Diseño y Arquitectura del Modelo](#6-diseño-y-arquitectura-del-modelo)
7. [Entrenamiento del Modelo](#7-entrenamiento-del-modelo)
8. [Evaluación y Métricas](#8-evaluación-y-métricas)
9. [Interpretación de Resultados](#9-interpretación-de-resultados)
10. [Conclusiones y Recomendaciones de Negocio](#10-conclusiones-y-recomendaciones-de-negocio)
11. [Referencias](#11-referencias)


---
## 1. Resumen Ejecutivo

El Sistema Nacional de Pensiones (Ley 19990), administrado por la Oficina de Normalización Previsional (ONP), gestiona un padrón de más de 2.7 millones de pensionistas. Revisar este volumen mediante reglas fijas o inspección manual es inviable en tiempo y costo, por lo que este proyecto propone un pipeline de auditoría automatizado basado en Deep Learning.

Se implementaron **dos técnicas de Deep Learning complementarias**:

1. Un **Autoencoder no supervisado** que aprende el patrón de "normalidad" previsional a partir de ~2.73 millones de registros y calcula un **error de reconstrucción** por pensionista como score de anomalía.
2. Un **MLP (Multi-Layer Perceptron) Clasificador supervisado**, entrenado con pseudo-etiquetas derivadas del Autoencoder (percentil 99 del error de reconstrucción), que valida si el patrón de anomalía detectado es aprendible de forma directa y consistente.

El pipeline completo —ingesta, limpieza, transformación, entrenamiento, evaluación y exportación de resultados— se ejecutó en la nube (Google Colab), procesando 4 archivos CSV (cortes trimestrales de 2025) hasta un reporte ejecutivo en Excel, auditable por personal no técnico.

**Principales hallazgos:** el Autoencoder aisló el 1% de registros más atípicos (27,288 casos), y el MLP Clasificador confirmó sobre un conjunto de prueba independiente que ese patrón es matemáticamente consistente (AUC-ROC = 0.9999, recall = 99.9%). El hallazgo forense principal es un clúster de registros con contradicción lógica entre modalidad de jubilación y tipo de prestación (p. ej. "Jubilación General/Normal" combinada con el código de "Invalidez Especial / Capital Defunción"), compatible con posibles duplicidades de identidad o errores de migración de datos.

**Impacto esperado en el negocio:** el pipeline puede integrarse como filtro mensual previo a la emisión de planillas, priorizando auditorías de campo sobre los casos de mayor riesgo y reduciendo el costo de revisar manualmente el padrón completo.

---


## 2. Configuración del Entorno

### 2.1 Verificación de GPU (Recomendado para Deep Learning)

In [ ]:
# Verificar si hay GPU disponible
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU disponible: {gpus[0].name}")
else:
    print("⚠️ GPU no disponible. Usando CPU.")
    print("   Recomendación: En Colab, vaya a Runtime > Change runtime type > GPU")

print(f"\nTensorFlow version: {tf.__version__}")


### 2.2 Instalación de Librerías Adicionales (si es necesario)

In [ ]:
# Descomente e instale las librerías adicionales que necesite
# !pip install openpyxl
# !pip install shap


### 2.3 Importación de Librerías

In [ ]:
# =====================================================
# LIBRERÍAS FUNDAMENTALES
# =====================================================

# Manipulación de datos
import numpy as np
import pandas as pd
import glob
import os
import gc

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Deep Learning - TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Preprocesamiento y métricas
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Utilidades
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('ggplot')
sns.set_palette('viridis')
%matplotlib inline

# Semilla para reproducibilidad
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("✅ Todas las librerías importadas correctamente")
print(f"   TensorFlow version: {tf.__version__}")


### 2.4 Conexión con Google Drive (para cargar datos)

In [ ]:
# Montar Google Drive para acceder a los datos
from google.colab import drive
drive.mount('/content/drive')

# Ruta base del proyecto (dataset del padrón de pensionistas SNP)
BASE_PATH = '/content/drive/MyDrive/PROYECTO_SNP_GRUPO1/DataSet_SNP'

print(f"✅ Google Drive montado")
print(f"   Ruta base del proyecto: {BASE_PATH}")


---
## 3. Definición del Problema de Negocio

### 3.1 Contexto del Negocio

**Industria/Sector:** Seguridad social y previsión pública (pensiones).

**Empresa o caso de estudio:** Oficina de Normalización Previsional (ONP) del Perú, entidad encargada de administrar el Sistema Nacional de Pensiones (Ley N.º 19990).

**Situación actual:** la ONP administra un padrón masivo y heterogéneo de más de 2.7 millones de pensionistas. La revisión de este padrón mediante reglas fijas o inspección humana resulta inviable en tiempo y costo, y no escala frente a nuevos cortes de datos trimestrales.

---

### 3.2 Problema a Resolver

El reto técnico central es diseñar un método capaz de **detectar comportamientos atípicos (posible fraude o errores estructurales de datos) sin contar previamente con ejemplos etiquetados de fraude confirmado**. Es importante resolverlo porque:
- El costo de auditar manualmente 2.7 millones de registros es prohibitivo.
- Errores de migración de datos o duplicidades de identidad pueden generar pagos indebidos sostenidos en el tiempo.
- No existe un dataset histórico con casos de fraude ya etiquetados, lo que descarta un enfoque puramente supervisado desde el inicio.

---

### 3.3 Objetivos del Proyecto

**Objetivo General:**
Desarrollar un pipeline de Inteligencia Artificial basado en Deep Learning (Autoencoder + MLP Clasificador) para auditar automáticamente la totalidad del padrón de pensionistas del SNP y priorizar los casos de mayor riesgo de anomalía o fraude.

**Objetivos Específicos:**
1. Procesar y consolidar un dataset real de más de 2.7 millones de registros (4 archivos CSV trimestrales), resguardando la trazabilidad de los identificadores de pensionista.
2. Transformar variables categóricas y numéricas a un espacio matemático homogéneo (One-Hot Encoding y Min-Max Scaling) apto para redes neuronales.
3. Entrenar un modelo no supervisado (Autoencoder) que aprenda el patrón de normalidad y genere un score de anomalía por registro.
4. Entrenar un segundo modelo supervisado (MLP Clasificador) que valide el mismo patrón de anomalía, comparando ambos paradigmas con métricas estándar de clasificación.
5. Establecer un umbral estadístico objetivo (percentil 99) que aísle el subconjunto de registros más sospechosos.
6. Traducir las salidas matemáticas del modelo a un reporte ejecutivo en Excel, auditable por personal no técnico.

---

### 3.4 Tipo de Problema de Machine Learning

- [x] Detección de anomalías (no supervisado) — **Autoencoder**
- [x] Clasificación binaria (supervisado, sobre pseudo-etiquetas) — **MLP Clasificador**

**Justificación:** al no existir etiquetas confiables de fraude, se combina un enfoque no supervisado (Autoencoder) que descubre el patrón de normalidad, con un enfoque supervisado (MLP) entrenado sobre pseudo-etiquetas generadas por el propio Autoencoder. Esto permite validar si el patrón detectado es matemáticamente consistente y no ruido aleatorio, cumpliendo además el requisito de implementar como mínimo dos técnicas de Deep Learning.

---


---
## 4. Carga y Exploración de Datos

### 4.1 Carga de Datos

**Fuente:** 4 archivos CSV correspondientes a cortes trimestrales del padrón de pensionistas bajo la Ley 19990 (marzo, junio, setiembre y diciembre de 2025).

In [ ]:
# =====================================================
# CARGA DE DATOS
# =====================================================

# Buscar todos los archivos .csv en la carpeta del dataset
archivos_csv = glob.glob(os.path.join(BASE_PATH, "*.csv"))

print(f"Se encontraron {len(archivos_csv)} archivos. Iniciando lectura...")

# Leer y consolidar los 4 archivos
df_list = []
for archivo in archivos_csv:
    nombre_archivo = os.path.basename(archivo)
    print(f"Cargando: {nombre_archivo}...")
    df_temp = pd.read_csv(archivo, low_memory=False)  # low_memory=False evita alertas con archivos pesados
    df_list.append(df_temp)

# Unir todo en un solo dataset
df_total = pd.concat(df_list, ignore_index=True)

print(f"\n✅ Dataset cargado exitosamente")
print(f"   Dimensiones: {df_total.shape[0]:,} filas × {df_total.shape[1]} columnas")


### 4.2 Descripción del Dataset

| Variable | Tipo | Descripción |
|----------|------|-------------|
| id_persona | identificador | Identificador único del pensionista (se descarta antes de entrenar) |
| edadpen | numérica | Edad del pensionista |
| anios_aporte | numérica | Años de aportación al sistema |
| pension | numérica | Monto de pensión |
| bonif1...bonif9 | numérica | Bonificaciones y prestaciones secundarias |
| sexopen | categórica | Sexo del pensionista |
| estcivil | categórica | Estado civil |
| dpto | categórica | Departamento (código UBIGEO) |
| modalidad | categórica | Modalidad de jubilación |
| prestacion, prestacion2, prestacion3, prestacion4 | categórica | Tipo(s) de prestación otorgada |
| proporcional | categórica | Indicador de pensión proporcional |
| fnacpen, fnaccon | fecha | Fechas de nacimiento (se descartan por ser ruido para la red) |

No existe una variable "target" original: el problema es de **detección de anomalías no supervisada**, por lo que la pseudo-etiqueta se construye más adelante a partir del propio modelo (Sección 6).

---

### 4.3 Exploración Inicial de Datos (EDA)

In [ ]:
# =====================================================
# INFORMACIÓN GENERAL DEL DATASET
# =====================================================
print("=" * 60)
print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 60)

print("\n📊 Primeras 5 filas:")
display(df_total.head())

print("\n📋 Información del Dataset:")
print(df_total.info())

print("\n📈 Estadísticas Descriptivas:")
display(df_total.describe())


In [ ]:
# =====================================================
# ANÁLISIS DE VALORES FALTANTES
# =====================================================
print("=" * 60)
print("ANÁLISIS DE VALORES FALTANTES")
print("=" * 60)

missing_data = pd.DataFrame({
    'Total Faltantes': df_total.isnull().sum(),
    'Porcentaje (%)': (df_total.isnull().sum() / len(df_total) * 100).round(2)
})
missing_data = missing_data[missing_data['Total Faltantes'] > 0].sort_values('Porcentaje (%)', ascending=False)

if len(missing_data) > 0:
    print("\n⚠️ Variables con valores faltantes:")
    display(missing_data)
else:
    print("\n✅ No hay valores faltantes en el dataset")


In [ ]:
# =====================================================
# MATRIZ DE CORRELACIÓN DE VARIABLES NUMÉRICAS
# =====================================================
print("Generando Matriz de Correlación...")

# Solo columnas numéricas relevantes (las categóricas en One-Hot ensucian mucho este gráfico)
columnas_numericas = ['edadpen', 'anios_aporte', 'pension', 'bonif3', 'bonif5', 'bonif6', 'bonif8']
columnas_numericas = [col for col in columnas_numericas if col in df_total.columns]

matriz_correlacion = df_total[columnas_numericas].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_correlacion, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matriz de Correlación de Variables Numéricas")
plt.tight_layout()
plt.show()


In [ ]:
# =====================================================
# VISUALIZACIONES ADICIONALES DE EXPLORACIÓN
# =====================================================
print("Generando visualizaciones adicionales (dataset de ~2.7 millones de registros)...")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Gráfico 1: Distribución de las Pensiones
sns.histplot(data=df_total, x='pension', bins=50, ax=axes[0], color='#2ecc71')
axes[0].set_title('Distribución de los Montos de Pensión')
axes[0].set_xlabel('Monto de Pensión')
axes[0].set_ylabel('Cantidad de Pensionistas')

# Gráfico 2: Relación entre Edad y Años de Aporte (muestra 5% para no saturar la RAM)
muestra_grafico = df_total.sample(frac=0.05, random_state=RANDOM_SEED)
sns.scatterplot(data=muestra_grafico, x='edadpen', y='anios_aporte',
                alpha=0.3, ax=axes[1], color='#3498db')
axes[1].set_title('Relación: Edad vs Años de Aporte (Muestra 5%)')
axes[1].set_xlabel('Edad')
axes[1].set_ylabel('Años Aportados')

# Gráfico 3: Cantidad de pensionistas por Modalidad
sns.countplot(data=df_total, x='modalidad', ax=axes[2], palette='magma')
axes[2].set_title('Concentración por Modalidad')
axes[2].set_xlabel('Código de Modalidad')
axes[2].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()


### 4.4 Hallazgos del EDA

**Hallazgos Principales:**
1. La matriz de correlación de Pearson no muestra ninguna correlación perfecta (1.0) entre variables numéricas, lo que confirma que cada atributo aporta información única y justifica conservarlas todas para el modelo.
2. Existen valores faltantes en columnas de bonificaciones/prestaciones secundarias (vacío = "no aplica", no un dato perdido) y en años de aporte/modalidad.
3. Las variables categóricas (sexo, estado civil, departamento, modalidad, prestación) requieren codificación numérica antes de entrenar una red neuronal.

**Problemas Identificados:**
1. No existe una variable objetivo (target) etiquetada para "fraude", por lo que no se puede aplicar clasificación supervisada directa desde el inicio.
2. Las columnas `fnacpen` y `fnaccon` (fechas de nacimiento) y `id_persona` no aportan valor predictivo directo y deben tratarse como ruido o metadato de trazabilidad.

**Acciones a Tomar:**
1. Imputar nulos de forma diferenciada según su naturaleza (lógica vs. estadística).
2. Aplicar One-Hot Encoding + Min-Max Scaling y construir pseudo-etiquetas a partir de un modelo no supervisado (Sección 6).

---

---
## 5. Preprocesamiento de Datos

### 5.1 Tratamiento de Valores Faltantes

In [ ]:
# =====================================================
# FASE 1 — LIMPIEZA DE DATOS (Data Cleaning)
# =====================================================

# A. Llenar nulos lógicos (bonificaciones y prestaciones secundarias) con 0
cols_bonif = [f'bonif{i}' for i in range(1, 10)]
for col in cols_bonif + ['prestacion3', 'prestacion4']:
    if col in df_total.columns:
        df_total[col] = df_total[col].fillna(0)

# B. Imputación estadística (mediana para numéricas, moda para categóricas)
if 'anios_aporte' in df_total.columns:
    mediana_aporte = df_total['anios_aporte'].median()
    df_total['anios_aporte'] = df_total['anios_aporte'].fillna(mediana_aporte)

if 'modalidad' in df_total.columns:
    moda_modalidad = df_total['modalidad'].mode()[0]
    df_total['modalidad'] = df_total['modalidad'].fillna(moda_modalidad)

# C. Eliminar columnas que son "ruido" para la Red Neuronal
cols_a_eliminar = ['id_persona', 'fnacpen', 'fnaccon']
cols_a_eliminar = [col for col in cols_a_eliminar if col in df_total.columns]
df_limpio = df_total.drop(columns=cols_a_eliminar)

# D. Eliminar las pocas filas que tengan vacíos críticos remanentes (ej. departamento)
df_limpio = df_limpio.dropna()

print("--- REPORTE FINAL DE LA FASE 1 (LIMPIEZA) ---")
print(f"Registros listos (sin nulos): {df_limpio.shape[0]:,}")
print(f"Variables (columnas) útiles: {df_limpio.shape[1]}")


### 5.2 Codificación de Variables Categóricas y Escalado

In [ ]:
# =====================================================
# FASE 2 — TRANSFORMACIÓN MATEMÁTICA (Feature Engineering)
# =====================================================
print("Iniciando Fase 2: Transformación de variables...")

# 1. Separar variables por su naturaleza
vars_numericas = ['edadpen', 'anios_aporte', 'pension', 'redadpen']
vars_numericas.extend(cols_bonif)
vars_numericas = [col for col in vars_numericas if col in df_limpio.columns]

vars_categoricas = ['sexopen', 'estcivil', 'dpto', 'modalidad', 'prestacion',
                     'prestacion2', 'prestacion3', 'prestacion4', 'proporcional']
vars_categoricas = [col for col in vars_categoricas if col in df_limpio.columns]

# 2. Transformación de categóricas: One-Hot Encoding
# Convierte clases en columnas binarias (0/1), evitando que la red interprete jerarquías falsas
df_transformado = pd.get_dummies(df_limpio, columns=vars_categoricas, drop_first=True)

# 3. Escalamiento de numéricas: Min-Max Scaling al rango [0, 1]
scaler = MinMaxScaler()
df_transformado[vars_numericas] = scaler.fit_transform(df_transformado[vars_numericas])

# 4. Asegurar formato numérico puro (float)
df_transformado = df_transformado.astype(float)

print("--- REPORTE FINAL DE LA FASE 2 (TRANSFORMACIÓN) ---")
print(f"Columnas originales: {df_limpio.shape[1]}")
print(f"Columnas transformadas (expandidas): {df_transformado.shape[1]}")
print("El dataset ahora es 100% matemático y está listo para las redes neuronales.")


### 5.3 Preparación de Datos para Deep Learning

El Autoencoder (Sección 6.2) se entrena sobre la totalidad de `df_transformado` de forma auto-supervisada (entrada = salida). El split train/validation/test se aplica más adelante, exclusivamente para el MLP Clasificador supervisado (Sección 6.3), una vez generadas las pseudo-etiquetas.

---
## 6. Diseño y Arquitectura del Modelo

### 6.1 Justificación de la Arquitectura

Se optó por combinar **dos arquitecturas de Deep Learning** en lugar de una sola, dado que no existían etiquetas de fraude confiables:

- **Técnica 1 — Autoencoder (no supervisado):** una red "cuello de botella" (encoder-decoder) que aprende a reconstruir el 100% de los registros normales. Los registros que el modelo reconstruye peor (mayor error) son los estadísticamente más atípicos. No requiere conocer el fraude de antemano.
- **Técnica 2 — MLP Clasificador (supervisado):** un Perceptrón Multicapa entrenado sobre pseudo-etiquetas (el 1% con mayor error de reconstrucción del Autoencoder = percentil 99). Permite validar si el patrón de anomalía es aprendible de forma directa y obtener métricas de clasificación estándar (Precision, Recall, F1, AUC) que el Autoencoder por sí solo no entrega.

El número de capas y neuronas de ambas redes se determinó de forma empírica, buscando un cuello de botella progresivo (Autoencoder: 32→16→8→16→32) y una arquitectura de clasificación con regularización (MLP: 64→32→16 con Dropout) suficiente para las ~66 variables transformadas, sin sobreajustar dado el fuerte desbalance de clases (~1% positivos).

### 6.2 Técnica 1: Arquitectura del Autoencoder (No Supervisado)

In [ ]:
# =====================================================
# TÉCNICA 1: ARQUITECTURA DEL AUTOENCODER
# =====================================================
print("Diseño de la Arquitectura del Autoencoder...")

# Cantidad exacta de variables de entrada (~66 columnas matemáticas)
input_dim = df_transformado.shape[1]

autoencoder = models.Sequential(name="Autoencoder_Deteccion_Anomalias")

# --- ENCODER (el compresor) ---
autoencoder.add(layers.Dense(32, activation='relu', input_shape=(input_dim,)))
autoencoder.add(layers.Dense(16, activation='relu'))

# --- ESPACIO LATENTE (cuello de botella): patrón más puro y concentrado de un pensionista ---
autoencoder.add(layers.Dense(8, activation='relu'))

# --- DECODER (el reconstructor) ---
autoencoder.add(layers.Dense(16, activation='relu'))
autoencoder.add(layers.Dense(32, activation='relu'))

# Capa de salida: mismo tamaño que la entrada; sigmoid porque los datos están escalados en [0,1]
autoencoder.add(layers.Dense(input_dim, activation='sigmoid'))

# Compilación: MSE mide qué tan distinta es la reconstrucción del dato original
autoencoder.compile(optimizer='adam', loss='mse')

print("=" * 60)
print("ARQUITECTURA DEL MODELO — AUTOENCODER")
print("=" * 60)
autoencoder.summary()


### 6.3 Técnica 2: Arquitectura del MLP Clasificador (Supervisado)

In [ ]:
# =====================================================
# TÉCNICA 2: ARQUITECTURA DEL MLP CLASIFICADOR
# =====================================================
# (Se construye una vez generadas las pseudo-etiquetas en la Sección 7.3)
print("Arquitectura del MLP Clasificador — se instancia en la Sección 7.3, tras construir las pseudo-etiquetas.")


### 6.4 Diagrama de la Arquitectura

```
TÉCNICA 1 — AUTOENCODER (no supervisado)
Input (~66)  -->  Dense(32,relu) --> Dense(16,relu) --> Dense(8,relu) [espacio latente]
             -->  Dense(16,relu) --> Dense(32,relu) --> Dense(~66,sigmoid) [reconstrucción]
Score de anomalía = Error de Reconstrucción (MSE por registro)

TÉCNICA 2 — MLP CLASIFICADOR (supervisado, sobre pseudo-etiquetas)
Input (~66)  -->  Dense(64,relu) --> Dropout(0.3) --> Dense(32,relu) --> Dropout(0.2)
             -->  Dense(16,relu) --> Dense(1,sigmoid) [probabilidad de anomalía]
```

---

---
## 7. Entrenamiento del Modelo

### 7.1 Preparación de Datos para el Autoencoder (Fase 4)

In [ ]:
# =====================================================
# FASE 4 — PASO 1: PREPARACIÓN DE DATOS PARA EL AUTOENCODER
# =====================================================
print("--- FASE 4: PASO 1 - Preparación de Datos ---")

# Conservamos una copia con IDs para trazabilidad forense posterior (Sección 9)
df_limpio_con_id = df_total.copy()
for col in cols_bonif + ['prestacion3', 'prestacion4']:
    if col in df_limpio_con_id.columns:
        df_limpio_con_id[col] = df_limpio_con_id[col].fillna(0)

if 'anios_aporte' in df_limpio_con_id.columns:
    df_limpio_con_id['anios_aporte'] = df_limpio_con_id['anios_aporte'].fillna(df_limpio_con_id['anios_aporte'].median())
if 'modalidad' in df_limpio_con_id.columns:
    df_limpio_con_id['modalidad'] = df_limpio_con_id['modalidad'].fillna(df_limpio_con_id['modalidad'].mode()[0])

df_limpio_con_id = df_limpio_con_id.drop(columns=['fnacpen', 'fnaccon'], errors='ignore').dropna()

# Truco de memoria: convertir a float32 para reducir a la mitad el consumo de RAM
X_train = df_transformado.astype('float32').values
gc.collect()
print("Datos listos y comprimidos en RAM.")


### 7.2 Entrenamiento del Autoencoder (Fase 4)

In [ ]:
# =====================================================
# FASE 4 — PASO 2: ENTRENAMIENTO DEL AUTOENCODER
# =====================================================
print("--- FASE 4: PASO 2 - Entrenamiento ---")

print("Entrenando red neuronal. Esto tomará un par de minutos...")
history = autoencoder.fit(
    X_train, X_train,          # entrada = salida objetivo (auto-supervisado)
    epochs=10,
    batch_size=2048,
    validation_split=0.2,
    shuffle=True,
    verbose=1
)
print("¡Entrenamiento finalizado con éxito!")


### 7.3 Construcción de Pseudo-Etiquetas y Entrenamiento del MLP Clasificador

In [ ]:
# =====================================================
# FASE 4 — PASOS 3 y 4: ERRORES DE RECONSTRUCCIÓN Y UMBRAL
# =====================================================
print("--- FASE 4: PASO 3 - Cálculo de Errores ---")

# Predecir por lotes para evitar que la RAM explote
reconstrucciones = autoencoder.predict(X_train, batch_size=2048)

# Error de reconstrucción por registro (MSE)
errores = np.mean(np.square(X_train - reconstrucciones), axis=1)

# Liberar RAM de variables pesadas
del reconstrucciones
gc.collect()

# Pegar el error al dataset con IDs
df_limpio_con_id['Error_Reconstruccion'] = errores

# Umbral de anomalía: percentil 99
umbral_anomalia = np.percentile(errores, 99)
print(f"Umbral de anomalía detectado: {umbral_anomalia:.4f}")

casos_sospechosos = df_limpio_con_id[df_limpio_con_id['Error_Reconstruccion'] > umbral_anomalia]
casos_sospechosos = casos_sospechosos.sort_values(by='Error_Reconstruccion', ascending=False)
print(f"¡Atención! Se detectaron {len(casos_sospechosos):,} registros altamente anómalos (Técnica 1).")


In [ ]:
# =====================================================
# TÉCNICA 2 — MLP CLASIFICADOR SUPERVISADO
# =====================================================
# ¿Por qué una segunda técnica? El Autoencoder es no supervisado: no usa etiquetas,
# solo aprende a reconstruir. Para cumplir el requisito de mínimo 2 técnicas de Deep
# Learning, se entrena ahora un MLP supervisado que aprende a clasificar directamente
# si un registro es "normal" (0) o "anómalo" (1), usando como pseudo-etiqueta el
# resultado del Autoencoder (percentil 99 del error de reconstrucción).

print("--- FASE 6: PASO 1 - Construcción de Pseudo-Etiquetas ---")

y_pseudo = (df_limpio_con_id['Error_Reconstruccion'] > umbral_anomalia).astype(int)
print(f"Registros normales (0): {(y_pseudo == 0).sum():,}")
print(f"Registros anómalos (1): {(y_pseudo == 1).sum():,}  ({y_pseudo.mean()*100:.2f}% del total)")

# Mismas variables transformadas que el Autoencoder
X_clf = df_transformado.astype('float32').values
y_clf = y_pseudo.values

# Split estratificado train/val/test para conservar la proporción del 1% anómalo
X_train_c, X_temp_c, y_train_c, y_temp_c = train_test_split(
    X_clf, y_clf, test_size=0.30, stratify=y_clf, random_state=RANDOM_SEED
)
X_val_c, X_test_c, y_val_c, y_test_c = train_test_split(
    X_temp_c, y_temp_c, test_size=0.50, stratify=y_temp_c, random_state=RANDOM_SEED
)
print(f"\nTrain: {X_train_c.shape[0]:,} | Val: {X_val_c.shape[0]:,} | Test: {X_test_c.shape[0]:,}")


In [ ]:
print("--- FASE 6: PASO 2 - Arquitectura del MLP Clasificador ---")

input_dim_clf = X_train_c.shape[1]

mlp_clf = models.Sequential(name="MLP_Clasificador_Anomalias")
mlp_clf.add(layers.Input(shape=(input_dim_clf,)))
mlp_clf.add(layers.Dense(64, activation='relu'))
mlp_clf.add(layers.Dropout(0.3))
mlp_clf.add(layers.Dense(32, activation='relu'))
mlp_clf.add(layers.Dropout(0.2))
mlp_clf.add(layers.Dense(16, activation='relu'))
mlp_clf.add(layers.Dense(1, activation='sigmoid'))  # salida binaria

mlp_clf.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

mlp_clf.summary()

# Clases MUY desbalanceadas (~1% positivos) → ponderación de clases
n_neg = (y_train_c == 0).sum()
n_pos = (y_train_c == 1).sum()
class_weight = {0: 1.0, 1: n_neg / n_pos}
print(f"\nClass weight aplicado a la clase minoritaria (anómalos): {class_weight[1]:.1f}")


In [ ]:
print("--- FASE 6: PASO 3 - Entrenamiento del MLP Clasificador ---")

early_stop = callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True)

history_clf = mlp_clf.fit(
    X_train_c, y_train_c,
    validation_data=(X_val_c, y_val_c),
    epochs=30,
    batch_size=2048,
    class_weight=class_weight,
    callbacks=[early_stop],
    shuffle=True,
    verbose=1
)
print("¡Entrenamiento del MLP Clasificador finalizado!")


### 7.4 Visualización del Entrenamiento

In [ ]:
# =====================================================
# CURVA DE APRENDIZAJE — AUTOENCODER
# =====================================================
print("--- FASE 4: PASO 6 - Gráfico de Aprendizaje (Autoencoder) ---")
plt.plot(history.history['loss'], label='Error de Entrenamiento')
plt.plot(history.history['val_loss'], label='Error de Validación')
plt.title('Curva de Aprendizaje del Autoencoder')
plt.ylabel('Error (Loss)')
plt.xlabel('Epoch (Pasadas)')
plt.legend()
plt.show()


---
## 8. Evaluación y Métricas

### 8.1 Evaluación de la Técnica 1: Distribución del Error de Reconstrucción (Autoencoder)

In [ ]:
# =====================================================
# GRÁFICO FINAL DE DISTRIBUCIÓN DEL ERROR DE RECONSTRUCCIÓN
# =====================================================
print("Generando gráfico final de distribución...")

plt.figure(figsize=(12, 6))
sns.histplot(df_limpio_con_id['Error_Reconstruccion'], bins=100, color='#3498db', log_scale=(False, True))

plt.axvline(umbral_anomalia, color='#e74c3c', linestyle='dashed', linewidth=3,
            label=f'Línea de Alarma (Top 1%: > {umbral_anomalia:.4f})')

plt.title('Separación del Modelo: Normalidad vs. Anomalía Severa', fontsize=14)
plt.xlabel('Nivel de Error de Reconstrucción (Rareza)', fontsize=12)
plt.ylabel('Cantidad de Pensionistas (Escala Logarítmica)', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()


### 8.2 Evaluación de la Técnica 2: MLP Clasificador (Test Set)

In [ ]:
print("--- FASE 6: PASO 4 - Evaluación del MLP Clasificador ---")

y_pred_proba = mlp_clf.predict(X_test_c, batch_size=2048).ravel()
y_pred_label = (y_pred_proba >= 0.5).astype(int)

print("\nReporte de Clasificación (Test set):")
print(classification_report(y_test_c, y_pred_label, target_names=['Normal', 'Anómalo'], digits=3))

auc_score = roc_auc_score(y_test_c, y_pred_proba)
print(f"AUC-ROC: {auc_score:.4f}")

cm = confusion_matrix(y_test_c, y_pred_label)
print("\nMatriz de Confusión:")
print(cm)

# Curva ROC
fpr, tpr, _ = roc_curve(y_test_c, y_pred_proba)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'MLP Clasificador (AUC = {auc_score:.3f})', color='#3498db', linewidth=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Azar (AUC = 0.5)')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC — MLP Clasificador Supervisado vs. Autoencoder (pseudo-etiquetas)')
plt.legend()
plt.tight_layout()
plt.show()


### 8.3 Comparación entre las dos Técnicas de Deep Learning

| Aspecto | Técnica 1: Autoencoder | Técnica 2: MLP Clasificador |
|---|---|---|
| Tipo de aprendizaje | No supervisado | Supervisado (con pseudo-etiquetas del Autoencoder) |
| Objetivo | Reconstruir la entrada | Clasificar Normal vs. Anómalo |
| Salida | Error de reconstrucción (continuo) | Probabilidad de anomalía (0 a 1) |
| Ventaja | No requiere conocer el fraude de antemano | Probabilidad interpretable; permite ajustar el umbral de decisión (trade-off precisión/recall) |
| Uso recomendado | Descubrimiento inicial de patrones desconocidos | Producción/monitoreo continuo, una vez validado el patrón por auditoría de campo |

Un AUC-ROC del MLP Clasificador de 0.9999 sobre datos de prueba nunca vistos en entrenamiento **valida matemáticamente** que el patrón de anomalía detectado por el Autoencoder es aprendible y consistente, no ruido aleatorio. Ambos modelos forman en conjunto un pipeline robusto: el Autoencoder descubre el patrón sin supervisión y el MLP lo aprende a clasificar de forma eficiente para producción.

---

---
## 9. Interpretación de Resultados

### 9.1 Traducción del Reporte para Auditoría no Técnica

In [ ]:
import pandas as pd

print("--- FASE 5: REPORTE FINAL Y TRADUCCIÓN ---")

diccionario_dpto = {
    1.0: 'Amazonas', 2.0: 'Áncash', 3.0: 'Apurímac', 4.0: 'Arequipa',
    5.0: 'Ayacucho', 6.0: 'Cajamarca', 7.0: 'Callao', 8.0: 'Cusco',
    9.0: 'Huancavelica', 10.0: 'Huánuco', 11.0: 'Ica', 12.0: 'Junín',
    13.0: 'La Libertad', 14.0: 'Lambayeque', 15.0: 'Lima', 16.0: 'Loreto',
    17.0: 'Madre de Dios', 18.0: 'Moquegua', 19.0: 'Pasco', 20.0: 'Piura',
    21.0: 'Puno', 22.0: 'San Martín', 23.0: 'Tacna', 24.0: 'Tumbes', 25.0: 'Ucayali'
}
diccionario_modalidad = {
    1.0: 'Jubilación General/Normal', 2.0: 'Jubilación Adelantada',
    3.0: 'Régimen Especial', 4.0: 'Jubilación Minera',
    5.0: 'Jubilación Construcción Civil', 6.0: 'Jubilación Marítima/Pesquera',
    7.0: 'Otros Regímenes'
}
diccionario_prestacion = {
    1.0: 'Titular (Jubilación)', 2.0: 'Viudez', 3.0: 'Orfandad',
    4.0: 'Ascendencia (Padres)', 5.0: 'Invalidez',
    6.0: 'Invalidez Especial / Capital Defunción', 7.0: 'Otros Beneficios'
}

casos_sospechosos['Nombre_Dpto'] = casos_sospechosos['dpto'].map(diccionario_dpto).fillna('Desconocido')
casos_sospechosos['Tipo_Modalidad'] = casos_sospechosos['modalidad'].map(diccionario_modalidad).fillna('Desconocido')
casos_sospechosos['Tipo_Prestacion'] = casos_sospechosos['prestacion'].map(diccionario_prestacion).fillna('Desconocido')

# Exportar a Excel para auditoría de campo
nombre_archivo_salida = 'Pensionistas_Anomalos_Detectados.xlsx'
casos_sospechosos.to_excel(nombre_archivo_salida, index=False, engine='openpyxl')
print(f"Archivo de resultados guardado como: {nombre_archivo_salida}")

print("\nTop 5 casos más anómalos:")
display(casos_sospechosos[['Nombre_Dpto', 'Tipo_Modalidad', 'Tipo_Prestacion', 'Error_Reconstruccion']].head())


### 9.2 Hallazgos Forenses Principales

**El "Clúster San Martín": evidencia de posible clonación o fraude.** Los primeros 4 registros del ranking de anomalías comparten el mismo error matemático exacto y montos de pensión idénticos, con identificadores distintos. Esto indica, con alta probabilidad, que se trata de la **misma persona física registrada múltiples veces** en el sistema — ya sea por un error de migración de datos (libretas electorales antiguas y DNIs duplicados) o por un fraude sistemático de identidades variantes.

**La contradicción legal (el detonante de la IA).** La razón principal por la que la red neuronal aisló estos casos del resto de los 2.7 millones de perfiles es la incompatibilidad de sus reglas de negocio: modalidad **"Jubilación General/Normal"** combinada con prestación **"Invalidez Especial / Capital Defunción"**. El modelo aprendió que el 99% de las personas con jubilación normal están vivas y cobran por sus años de trabajo; sin embargo, a estos casos se les asignó un código de pago reservado para defunción o invalidez especial — una contradicción lógica que ninguna regla fija habría detectado sin revisar el cruce exacto de ambas variables.

**El caso aislado de La Libertad.** Un quinto caso, en una persona de 97 años, presenta la misma contradicción modalidad-prestación y el mismo monto de pensión que el clúster de San Martín, pero en otra región. Esto sugiere que el error de códigos incompatibles no es un problema aislado de una sola oficina, sino una vulnerabilidad o error de digitación presente en distintas regiones del país, que afecta particularmente a la población más anciana del sistema.

### 9.3 Interpretación de Negocio

**Insights Principales:**
1. El pipeline permite priorizar auditorías de campo sobre un subconjunto acotado (27,288 casos) en lugar de revisar manualmente 2.7 millones de registros.
2. El patrón de mayor riesgo detectado no es solo "fraude" en sentido estricto, sino también **fallas estructurales de la base de datos** (contradicciones entre modalidad y prestación, posibles duplicidades de identidad).
3. El AUC-ROC de 0.9999 del MLP Clasificador demuestra que el patrón de anomalía es matemáticamente estable y replicable, no un artefacto del Autoencoder.

**Factores más importantes:** la combinación `modalidad de jubilación` × `tipo de prestación` resultó ser la señal más discriminante del patrón de anomalía, junto con la coincidencia exacta de montos de pensión entre distintos identificadores.

**Patrones identificados:** existencia de un clúster geográfico (San Martín) con múltiples identificadores para lo que aparenta ser la misma persona, y una vulnerabilidad de codificación (modalidad-prestación) que se repite en al menos otra región (La Libertad), lo que sugiere un problema sistémico y no un caso aislado.

---

---
## 10. Conclusiones y Recomendaciones de Negocio

### 10.1 Resumen de Resultados

El pipeline de dos técnicas de Deep Learning procesó exitosamente 2,729,525 registros del padrón de pensionistas del SNP. El Autoencoder aisló el 1% de mayor error de reconstrucción (27,288 casos) sin necesidad de etiquetas de fraude previas, y el MLP Clasificador confirmó sobre un conjunto de prueba independiente (409,429 registros nunca vistos en entrenamiento) que ese patrón es aprendible con AUC-ROC = 0.9999 y recall = 99.9%. El análisis forense de los casos de mayor riesgo reveló tanto posibles duplicidades de identidad como una contradicción sistemática entre modalidad de jubilación y tipo de prestación.

### 10.2 Conclusiones

1. **Eficacia del enfoque no supervisado:** el Autoencoder demostró capacidad para procesar más de 2.7 millones de registros y aislar irregularidades (loss final ≈ 0.0096) sin etiquetas previas de fraude, validando este enfoque como punto de partida cuando no existe historial etiquetado.
2. **Validación cruzada confirmada:** el MLP Clasificador, entrenado sobre las pseudo-etiquetas del Autoencoder, confirma con evidencia sobre un conjunto de prueba independiente (AUC-ROC = 0.9999, recall = 99.9%) que el patrón de anomalía es matemáticamente consistente.
3. **Detección de fraude y de fallas estructurales:** el pipeline detecta tanto posibles conductas fraudulentas como fallas estructurales de la base de datos de la ONP (contradicción modalidad-prestación, posibles identidades múltiples activas para un mismo individuo).
4. **Sensibilidad del hallazgo al corte de datos:** el patrón dominante en el Top 5 de anomalías puede variar entre corridas con distintos cortes de datos, lo que demuestra que el 1% de mayor error de reconstrucción puede capturar más de un tipo de anomalía simultáneamente.
5. **Escalabilidad del pipeline:** el diseño modular por fases y las optimizaciones de memoria (float32, batching, `gc.collect()`) permiten ejecutar el pipeline completo con hardware estándar (una sola GPU de Colab), facilitando su escalabilidad y mantenimiento.

### 10.3 Recomendaciones de Negocio

**Recomendaciones a Corto Plazo:**
1. Desplegar un equipo de fiscalización para revisar en campo los expedientes físicos de los identificadores aislados (priorizando los casos con contradicción modalidad-prestación y posible duplicidad de identidad).
2. Auditar una muestra representativa del conjunto completo de 27,288 casos exportados —no solo el Top 5— para caracterizar la proporción real de cada tipo de anomalía.

**Recomendaciones a Mediano Plazo:**
1. Integrar este pipeline (Autoencoder + MLP Clasificador) como filtro previo mensual a la emisión oficial de planillas de la ONP, reentrenando periódicamente con nuevos datos validados por auditoría.
2. Establecer un umbral de re-entrenamiento cuando el AUC del MLP Clasificador sobre nuevos datos caiga por debajo de un valor de referencia (p. ej. 0.995), como señal de *data drift*.

**Recomendaciones a Largo Plazo:**
1. Establecer candados lógicos en el sistema transaccional de origen que impidan combinaciones contradictorias entre modalidad de jubilación y tipo de prestación.
2. Implementar controles de deduplicación de identidad en la etapa de migración de datos, para prevenir la aparición de identificadores múltiples asociados a una misma persona.

### 10.4 Limitaciones del Estudio

1. Las pseudo-etiquetas usadas para entrenar el MLP Clasificador provienen del propio Autoencoder, por lo que no constituyen una validación externa independiente del fraude real (aún pendiente de confirmación por auditoría de campo).
2. El umbral del percentil 99 es una decisión estadística, no un criterio legal o de negocio validado formalmente por la ONP.
3. El dataset cubre únicamente cortes trimestrales de un año (2025), lo que limita el análisis de tendencias de mediano/largo plazo en los patrones de anomalía.

### 10.5 Trabajo Futuro

1. Incorporar retroalimentación de auditorías de campo como etiquetas reales de fraude confirmado, para reentrenar el MLP Clasificador de forma totalmente supervisada.
2. Explorar arquitecturas adicionales (p. ej. Variational Autoencoders o modelos de grafos para detectar redes de identidades relacionadas).
3. Extender el pipeline a los regímenes DL 20530 (NSP_F0010) y otros programas de pensiones administrados por la ONP.

---

---
## 11. Referencias

1. Ley N.º 19990 — Sistema Nacional de Pensiones. Oficina de Normalización Previsional (ONP).
2. Codificación UBIGEO departamental — Instituto Nacional de Estadística e Informática (INEI).
3. Chollet, F. (2021). *Deep Learning with Python* (2nd ed.). Manning Publications.
4. Documentación oficial de TensorFlow/Keras: https://www.tensorflow.org/guide/keras
5. Notebook de entrenamiento completo (Autoencoder + MLP Clasificador), carpeta del modelo exportado y reporte Excel de anomalías (`Pensionistas_Anomalos_Detectados.xlsx`) se adjuntan como entregables complementarios a este informe.

---

---
## Anexos

### A. Guardado de los Modelos Entrenados

In [ ]:
# =====================================================
# GUARDAR LOS MODELOS ENTRENADOS
# =====================================================
print("=" * 60)
print("GUARDADO DE LOS MODELOS")
print("=" * 60)

# Guardar Autoencoder (Técnica 1)
autoencoder.save('autoencoder_snp.keras')
print("✅ Autoencoder guardado en: autoencoder_snp.keras")

# Guardar MLP Clasificador (Técnica 2)
mlp_clf.save('mlp_clasificador_snp.keras')
print("✅ MLP Clasificador guardado en: mlp_clasificador_snp.keras")

# Guardar el scaler usado en el preprocesamiento
import joblib
joblib.dump(scaler, 'scaler_snp.pkl')
print("✅ Scaler guardado en: scaler_snp.pkl")


### B. Cargar Modelos Guardados (para Inferencia)

In [ ]:
# =====================================================
# CARGAR MODELOS GUARDADOS PARA INFERENCIA
# =====================================================
import joblib
from tensorflow import keras

autoencoder_cargado = keras.models.load_model('autoencoder_snp.keras')
mlp_clf_cargado = keras.models.load_model('mlp_clasificador_snp.keras')
scaler_cargado = joblib.load('scaler_snp.pkl')

print("✅ Modelos y scaler cargados correctamente. Listos para inferencia sobre nuevos cortes de datos.")
